# Molecular Dynamic Cutoff Distance Method

In [1]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from tqdm.notebook import tqdm
from mpl_toolkits.mplot3d import Axes3D
from abc import ABC, abstractmethod
import math
import re
from scipy.optimize import minimize
from itertools import product
import seaborn as sns
from matplotlib import animation
import time
%pip install -U numba numba-cuda cupy-cuda12x
import numba as nb
from numba import cuda
import cupy as cp

FORCE_MAT_TILE_DIM = 32
SEGMENT = 32
FORCE_MAT_REDUC_BLOCK_DIM = 512 # must be power of 2
FORCE_MAT_EXPLICIT_BLOCK_DIM = 512
SPACE_N_DIM = 3
CUTOFF_BLOCK_SIZE = 128

CELL_DIM = 20
CUTOFF_COEFF = 2.5 # convention is 2.5*sigma

IDENTITY = 0
HARMONIC = 1
REPULSIVE = 2
ATTRACTIVE = 3

BOX_SIZE_IDX = 0
K_IDX = 1
R0_IDX = 2
EPSILON_ATTRACTIVE_IDX = 3
EPSILON_REPULSIVE_IDX = 4
SIGMA_IDX = 5

NB_FLOAT32 = 32
NB_FLOAT64 = 64

DIV_BY_ZERO_GUARD = 1e-12

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 46.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 22.1 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.9/30.9 MB 77.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.9 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: llvmlite
    Found existing installation: llvmlite 0.43.0
    Uninstalling llvmlite-0.43.0:
      Successfully uninstalled llvmlite-0.43.0
  Attempting uninstall: cuda-core
    Found existing installation: cuda-core 0.3.2
    Uninstalling cuda-core-0.3.2:
      Successfully uninstalled cuda-core-0.3.2
  Attempting uninstall: numba
    Found existing installation: numba 0.60.0
    Uninstalling numba-0.60.0:
      Successfully uninstalled numba-0.60.0
  Attempting uninstall: numba-cuda
    Found existing installation: numba-cuda 0.22.2
    Uninstalling numba-cuda-0.22.2:
      Successfully uninstalled numba-cuda-0.22

## Initialization Work

In [2]:
@nb.njit
def apply_pbc(pos: np.ndarray | cp.ndarray | float, box_size: float) -> np.ndarray | cp.ndarray:
    """
    Wraps the chain inside the unit cell
    """
    return pos % box_size

@nb.njit
def assign_position(positions, new_positions, idx):
    positions[0, idx] = new_positions[0]
    positions[1, idx] = new_positions[1]
    positions[2, idx] = new_positions[2]

@nb.njit
def initialize_chain_numba(
    n_particles: int,
    box_size: float,
    r0: float,
    rng: np.random._generator.Generator,
    dtype=np.float32
) -> np.ndarray:
    """
    Randomly initialize atom positions by growing the chain
    """
    positions = np.zeros((3, n_particles), dtype=dtype)

    current_position = np.array([box_size / 2, box_size / 2, box_size / 2], dtype=dtype)
    assign_position(positions, current_position, 0)

    for i in range(1, n_particles):
        direction = rng.normal(size = 3).astype(dtype)
        norm = np.sqrt(direction[0]**2 + direction[1]**2 + direction[2]**2)
        direction /= norm

        next_position = current_position + r0 * direction
        next_position = apply_pbc(next_position, box_size).astype(dtype)
        assign_position(positions, next_position, i)
        current_position = next_position

    return positions

def initialize_velocities_cupy(
    n_particles: int,
    target_temperature: float, 
    mass: float,
    rng: cp.random._generator_api.Generator,
    kB=1.0,
    dtype=cp.float32
) -> cp.ndarray:
    """
    Initialize particle velocities by drawing from the Maxwell-Botzmann distribution at target temperature
    """
    sigma = cp.sqrt(kB * target_temperature / mass).astype(dtype)

    velocities = rng.standard_normal(
        size=(3, n_particles), 
        dtype=dtype
    )
    velocities = velocities * sigma

    # Remove center-of-mass velocity
    velocities -= cp.mean(velocities, axis=-1).reshape(3, 1)

    return velocities

## GPU Code

### Cell Label Kernel

In [3]:
@cuda.jit
def atom_label_kernel(
    labels, # global labels array (output)
    positions, # global positions array (input)
    cell_size: float, # size of each cell
    dim: int # number of cell in each dimension
):
    idx = cuda.grid(1)
    if idx >= positions.shape[1]:
        return
    x = positions[0, idx]
    y = positions[1, idx]
    z = positions[2, idx]
    cell_x = int(x // cell_size) % dim
    cell_y = int(y // cell_size) % dim
    cell_z = int(z // cell_size) % dim
    
    labels[idx] = cell_x + cell_y * dim + cell_z * dim * dim

### Force Computation Helper

In [4]:
@cuda.jit(device=True)
def harmonic_force_d(r, k, r0):
    return -k * (r - r0)

well_coeff = 2**(1/6)

@cuda.jit(device=True)
def repulsive_lj_force_d(r, sigma, epsilon):
    if r < well_coeff * sigma:
        if r < DIV_BY_ZERO_GUARD:
            r = DIV_BY_ZERO_GUARD
        sixth_pow = (sigma / r) ** 6
        return -24.0 * epsilon / r * (sixth_pow - 2.0 * sixth_pow * sixth_pow)
    else:
        return 0.0

@cuda.jit(device=True)
def attractive_lj_force_d(r, sigma, epsilon):
    if r < DIV_BY_ZERO_GUARD:
        r = DIV_BY_ZERO_GUARD
    sixth_pow = (sigma / r) ** 6
    return -24.0 * epsilon / r * (sixth_pow - 2.0 * sixth_pow * sixth_pow)

@cuda.jit(device=True)
def calc_force_d(distance, params, sep):
    if sep == HARMONIC:
        return harmonic_force_d(distance, params[K_IDX], params[R0_IDX])
    elif sep == REPULSIVE:
        return repulsive_lj_force_d(distance, params[SIGMA_IDX], params[EPSILON_REPULSIVE_IDX])
    elif sep >= ATTRACTIVE:
        return attractive_lj_force_d(distance, params[SIGMA_IDX], params[EPSILON_ATTRACTIVE_IDX])
    return 0.0

### Force Computation (Sorted and Unsorted)

In [5]:

@nb.njit
def minimum_image_1d(dx, box_size):
    return dx - box_size * math.floor((dx / box_size) + 0.5)

# positions, forces are unsorted, so that forces[i] corresponds to positions[i]
@cuda.jit
def calc_force_cutoff_gpu_unsorted(
    forces,
    cell_start,
    positions,
    org_idx,
    cell_size,
    n_cells,
    cutoff,
):
    i = cuda.grid(1)

    if i >= positions.shape[1]:
        return

    i_o = org_idx[i]

    xi = positions[0, i]
    yi = positions[1, i]
    zi = positions[2, i]

    cx = int((xi % box_size) // cell_size)
    cy = int((yi % box_size) // cell_size)
    cz = int((zi % box_size) // cell_size)

    force_x = 0.0
    force_y = 0.0
    force_z = 0.0

    for dz in range(-1, 2):
        nz = (cz + dz + n_cells) % n_cells

        for dy in range(-1, 2):
            ny = (cy + dy + n_cells) % n_cells

            for dx in range(-1, 2):
                nx = (cx + dx + n_cells) % n_cells

                nbr = nx + ny * n_cells + nz * n_cells * n_cells

                start = cell_start[nbr]
                end = cell_start[nbr + 1]

                for j in range(start, end):
                    if j == i:
                        continue

                    xj = positions[0, j]
                    yj = positions[1, j]
                    zj = positions[2, j]

                    xij = minimum_image_1d(xi - xj, box_size)
                    yij = minimum_image_1d(yi - yj, box_size)
                    zij = minimum_image_1d(zi - zj, box_size)

                    r2 = xij * xij + yij * yij + zij * zij
                    if r2 < DIV_BY_ZERO_GUARD * DIV_BY_ZERO_GUARD:
                        continue

                    rij = math.sqrt(r2)

                    if rij > cutoff:
                        continue

                    j_o = org_idx[j]
                    sep = abs(j_o - i_o)

                    fmag = calc_force_d(rij, const_params, sep)

                    force_x += fmag * xij / rij
                    force_y += fmag * yij / rij
                    force_z += fmag * zij / rij

    forces[0, i_o] = force_x
    forces[1, i_o] = force_y
    forces[2, i_o] = force_z


# positions, forces are sorted by original indices, so that forces[i_o] corresponds to positions[i_o]
@cuda.jit
def calc_force_cutoff_gpu_sorted(
    forces,
    cell_start,
    positions,
    org_idx,
    cell_size,
    n_cells,
    cutoff,
):
    params = cuda.const.array_like(const_params)
    i = cuda.grid(1)

    if i >= positions.shape[1]:
        return

    i_o = org_idx[i]

    xi = positions[0, i]
    yi = positions[1, i]
    zi = positions[2, i]

    cx = int((xi % box_size) // cell_size)
    cy = int((yi % box_size) // cell_size)
    cz = int((zi % box_size) // cell_size)

    force_x = 0.0
    force_y = 0.0
    force_z = 0.0

    for dz in range(-1, 2):
        nz = (cz + dz + n_cells) % n_cells

        for dy in range(-1, 2):
            ny = (cy + dy + n_cells) % n_cells

            for dx in range(-1, 2):
                nx = (cx + dx + n_cells) % n_cells

                nbr = nx + ny * n_cells + nz * n_cells * n_cells

                start = cell_start[nbr]
                end = cell_start[nbr + 1]

                for j in range(start, end):
                    if j == i:
                        continue

                    xj = positions[0, j]
                    yj = positions[1, j]
                    zj = positions[2, j]

                    xij = minimum_image_1d(xi - xj, params[BOX_SIZE_IDX])
                    yij = minimum_image_1d(yi - yj, params[BOX_SIZE_IDX])
                    zij = minimum_image_1d(zi - zj, params[BOX_SIZE_IDX])

                    r2 = xij * xij + yij * yij + zij * zij
                    if r2 < DIV_BY_ZERO_GUARD * DIV_BY_ZERO_GUARD:
                        continue

                    rij = math.sqrt(r2)

                    if rij > cutoff:
                        continue

                    j_o = org_idx[j]
                    sep = abs(j_o - i_o)

                    fmag = calc_force_d(rij, params, sep)

                    force_x += fmag * xij / rij
                    force_y += fmag * yij / rij
                    force_z += fmag * zij / rij

    forces[0, i] = force_x
    forces[1, i] = force_y
    forces[2, i] = force_z

In [63]:
def calc_force_cutoff_gpu_sorted_wrapper(forces, positions, org_idx):
    labels_d = cp.zeros(positions.shape[1], dtype=cp.int32)
    dim = CELL_DIM
    cell_size = const_params[BOX_SIZE_IDX] / dim
    cutoff = const_params[SIGMA_IDX] * CUTOFF_COEFF
    n_total_cells = dim ** 3
    grid_size = (positions.shape[1] + CUTOFF_BLOCK_SIZE - 1) // CUTOFF_BLOCK_SIZE

    # label stage
    atom_label_kernel[grid_size, CUTOFF_BLOCK_SIZE](labels_d, positions, cell_size, dim)
    cuda.synchronize()

    # sort + build cell_start stage
    order = cp.argsort(labels_d)
    labels_sorted = labels_d[order]
    prev_org_idx = org_idx.copy()
    positions[:] = positions[:, order]
    org_idx[:] = prev_org_idx[order]


    counts = cp.bincount(labels_sorted, minlength=n_total_cells)
    cell_start = cp.zeros(n_total_cells + 1, dtype=cp.int32)
    cell_start[1:] = cp.cumsum(counts)

    # force stage
    cell_start_nb = cuda.as_cuda_array(cell_start)
    positions_nb = cuda.as_cuda_array(positions)
    orig_idx_nb = cuda.as_cuda_array(org_idx)
    calc_force_cutoff_gpu_sorted[grid_size, CUTOFF_BLOCK_SIZE](
        forces,
        cell_start_nb,
        positions_nb,
        orig_idx_nb,
        cell_size,
        dim,
        cutoff
    )
    cuda.synchronize()
    return order

def calc_force_cutoff_gpu_unsorted_wrapper(forces, positions):
    labels_d = cp.zeros(positions.shape[1], dtype=cp.int32)
    dim = CELL_DIM
    cell_size = const_params[BOX_SIZE_IDX] / dim
    cutoff = const_params[SIGMA_IDX] * CUTOFF_COEFF
    n_total_cells = dim ** 3
    grid_size = (positions.shape[1] + CUTOFF_BLOCK_SIZE - 1) // CUTOFF_BLOCK_SIZE

    # label stage
    atom_label_kernel[grid_size, CUTOFF_BLOCK_SIZE](labels_d, positions, cell_size, dim)
    cuda.synchronize()

    # sort + build cell_start stage
    order = cp.argsort(labels_d)
    labels_sorted = labels_d[order]
    positions_sorted = cp.empty_like(positions)
    positions_sorted[:] = positions[:, order]

    counts = cp.bincount(labels_sorted, minlength=n_total_cells)
    cell_start = cp.zeros(n_total_cells + 1, dtype=cp.int32)
    cell_start[1:] = cp.cumsum(counts)

    # force stage
    cell_start_nb = cuda.as_cuda_array(cell_start)
    positions_sorted_nb = cuda.as_cuda_array(positions_sorted)
    orig_idx_nb = cuda.as_cuda_array(org_idx)
    calc_force_cutoff_gpu_unsorted[grid_size, CUTOFF_BLOCK_SIZE](
        forces,
        cell_start_nb,
        positions_sorted_nb,
        orig_idx_nb,
        cell_size,
        dim,
        cutoff
    )
    cuda.synchronize()

### Histogram Plotting Helper

In [7]:
def atoms_per_cell_histogram(labels_sorted, n_total_cells):
    counts = cp.bincount(labels_sorted, minlength=n_total_cells)
    counts_np = counts.get()

    # Plot histogram
    plt.figure(figsize=(10, 6))
    plt.hist(counts_np, bins=range(1, int(counts_np.max()) + 2), edgecolor='black', alpha=0.7)
    plt.xlabel('Number of atoms per cell')
    plt.ylabel('Frequency (number of cells)')
    plt.title('Distribution of Atoms in Cells')
    plt.grid(True, alpha=0.3)
    plt.show()

## CPU Code

### Force Helper

In [8]:
@nb.njit
def harmonic_force_h(r, k, r0):
    return -k * (r-r0)

well_coeff = 2**(1/6)
@nb.njit
def repulsive_lj_force_h(r, sigma, epsilon):
    if r < well_coeff*sigma:
        r = max(r, DIV_BY_ZERO_GUARD)
        sixth_pow = (sigma/r) ** 6
        return -24*epsilon/r * (sixth_pow - 2*(sixth_pow**2))
    else:
        return 0

@nb.njit
def attractive_lj_force_h(r, sigma, epsilon):
    r = max(r, DIV_BY_ZERO_GUARD)
    sixth_pow = (sigma/r) ** 6
    return -24*epsilon/r * (sixth_pow - 2*(sixth_pow**2))

@nb.njit
def calc_force_h(distance, params, type):
    if type == HARMONIC or type == -HARMONIC:
        return harmonic_force_h(distance, params[K_IDX], params[R0_IDX])
    if type == REPULSIVE or type == -REPULSIVE:
        return repulsive_lj_force_h(distance, params[SIGMA_IDX], params[EPSILON_REPULSIVE_IDX])
    if type >= ATTRACTIVE or type <= -ATTRACTIVE:
        return attractive_lj_force_h(distance, params[SIGMA_IDX], params[EPSILON_ATTRACTIVE_IDX])
    return 0

@nb.njit
def minimum_image_1d_h(dx, box_size):
    return dx - box_size * round(dx / box_size)

### Squential Version Labeled

In [9]:
@nb.njit
def calc_force_cutoff_sequential_org(
    forces, # global forces array (output)
    positions, # global positions array (input)
    cell_size,
    dim,
    cutoff
):
    n_particles = positions.shape[1]

    labels = np.zeros(n_particles, dtype=np.int32)
    for i in range(n_particles):
        x = positions[0, i]
        y = positions[1, i]
        z = positions[2, i]
        cell_x = int(x // cell_size) % dim
        cell_y = int(y // cell_size) % dim
        cell_z = int(z // cell_size) % dim
        
        labels[i] = cell_x + cell_y * dim + cell_z * dim * dim

    order = np.argsort(labels)
    labels_sorted = labels[order]
    positions_sorted = positions[:, order]
    org_idx = order
    n_total_cells = dim**3

    counts = np.bincount(labels_sorted, minlength=n_total_cells)
    cell_start = np.zeros(n_total_cells + 1, dtype=np.int32)
    cell_start[1:] = np.cumsum(counts)

    for i in range(n_particles):
        i_o = org_idx[i]

        xi = positions_sorted[0, i]
        yi = positions_sorted[1, i]
        zi = positions_sorted[2, i]

        cx = int((xi % box_size) // cell_size)
        cy = int((yi % box_size) // cell_size)
        cz = int((zi % box_size) // cell_size)

        force_x = 0.0
        force_y = 0.0
        force_z = 0.0

        for dz in range(-1, 2):
            nz = (cz + dz + dim) % dim

            for dy in range(-1, 2):
                ny = (cy + dy + dim) % dim

                for dx in range(-1, 2):
                    nx = (cx + dx + dim) % dim

                    nbr = nx + ny * dim + nz * dim * dim

                    start = cell_start[nbr]
                    end = cell_start[nbr + 1]

                    for j in range(start, end):
                        if j == i:
                            continue

                        xj = positions_sorted[0, j]
                        yj = positions_sorted[1, j]
                        zj = positions_sorted[2, j]

                        xij = minimum_image_1d_h(xi - xj, box_size)
                        yij = minimum_image_1d_h(yi - yj, box_size)
                        zij = minimum_image_1d_h(zi - zj, box_size)

                        r2 = xij * xij + yij * yij + zij * zij
                        if r2 < DIV_BY_ZERO_GUARD * DIV_BY_ZERO_GUARD:
                            continue

                        rij = math.sqrt(r2)

                        if rij > cutoff:
                            continue

                        j_o = org_idx[j]
                        sep = abs(j_o - i_o)

                        fmag = calc_force_h(rij, const_params, sep)

                        force_x += fmag * xij / rij
                        force_y += fmag * yij / rij
                        force_z += fmag * zij / rij

        forces[0, i_o] = force_x
        forces[1, i_o] = force_y
        forces[2, i_o] = force_z

In [10]:
def calc_force_cutoff_sequential_org_wrapper(forces, positions):
    dim = CELL_DIM
    cell_size = const_params[BOX_SIZE_IDX] / dim
    cutoff = const_params[SIGMA_IDX] * CUTOFF_COEFF
    calc_force_cutoff_sequential_org(forces, positions, cell_size, dim, cutoff)

### Naive Version

In [11]:
@nb.njit
def calc_force_cutoff_sequential_naive(
  forces: np.ndarray, # global forces array (output)
  positions: np.ndarray, # global positions array (input)
  cutoff: float
):
  forces.fill(0)
  params = const_params
  n_particles = positions.shape[1]
  for i in range(n_particles-1):
    for j in range(i+1, n_particles):
      distance = 0
      for k in range(SPACE_N_DIM):
        dx = minimum_image_1d_h(positions[k, i] - positions[k, j], params[BOX_SIZE_IDX])
        distance += dx**2
      distance = math.sqrt(distance)
      if distance > cutoff:
        continue
      magnitude = calc_force_h(distance, params, j - i)
      for k in range(SPACE_N_DIM):
        f = magnitude * minimum_image_1d_h(positions[k, i] - positions[k, j], params[BOX_SIZE_IDX]) / max(distance, DIV_BY_ZERO_GUARD)
        forces[k, i] += f
        forces[k, j] -= f

In [12]:
def calc_force_cutoff_sequential_naive_wrapper(forces, positions):
    calc_force_cutoff_sequential_naive(forces, positions, const_params[SIGMA_IDX] * CUTOFF_COEFF)

## Test Runtime

In [25]:
chain_lens = np.round(np.logspace(1, 5, 10)).astype(np.int32)
box_size = 100.0
dim = 20
cell_size = box_size / dim
n_total_cells = dim ** 3
block_size = 128
n_iters = 5

const_params_h = np.array([100, 300, 1, 0.5, 1.0, 1.0], dtype=np.float64)
const_params_d = cp.array(const_params_h)

time_df_gpu = []
time_df_cpu = []

for chain_len in tqdm(chain_lens):
    rng_np = np.random.default_rng(chain_len)
    pos = initialize_chain_numba(chain_len, box_size, 1.0, rng_np, dtype=np.float64)
    positions_d = cp.array(pos)
    forces_d = cp.zeros_like(positions_d)
    forces_h = np.zeros_like(pos, dtype=np.float64)

    # -------------------------
    # Warmup GPU
    # -------------------------

    calc_force_cutoff_gpu_unsorted_wrapper(forces_d, positions_d)

    # -------------------------
    # Warmup CPU
    # -------------------------
    calc_force_cutoff_sequential_org_wrapper(forces_h, pos)

    # -------------------------
    # Timed GPU
    # -------------------------
    for _ in range(n_iters):
        positions_d = cp.array(pos)
        forces_d = cp.zeros_like(positions_d)

        start_time = cuda.event(timing=True)
        end_time = cuda.event(timing=True)

        start_time.record()

        calc_force_cutoff_gpu_unsorted_wrapper(forces_d, positions_d)

        end_time.record()
        end_time.synchronize()

        time_df_gpu.append({
            "chain_len": int(chain_len),
            "total_time_ms": cuda.event_elapsed_time(start_time, end_time),
        })

    # -------------------------
    # Timed CPU
    # -------------------------
    for _ in range(n_iters):
        forces_h = np.zeros_like(pos, dtype=np.float64)

        start_time = time.perf_counter_ns()
        calc_force_cutoff_sequential_naive_wrapper(forces_h, pos)
        end_time = time.perf_counter_ns()

        time_df_cpu.append({
            "chain_len": int(chain_len),
            "total_time_ms": (end_time - start_time) / 1e6
        })

### Save Timing

In [26]:
time_df_gpu = pd.DataFrame(time_df_gpu)
time_df_cpu = pd.DataFrame(time_df_cpu)
time_df_gpu.to_csv('gpu_timing_2.csv', index=False)
time_df_cpu.to_csv('cpu_timing_2.csv', index=False)

### Read Timing

In [27]:
# Read csv
gpu_df = pd.read_csv("gpu_timing_2.csv")
cpu_df = pd.read_csv("cpu_timing_2.csv")

# Average repeated runs for each chain length
gpu_mean = (
    gpu_df.groupby("chain_len", as_index=False)[
        ["label_time_ms", "build_time_ms", "force_time_ms", "total_time_ms"]
    ]
    .mean()
    .sort_values("chain_len")
)

cpu_mean = (
    cpu_df.groupby("chain_len", as_index=False)[["total_time_ms"]]
    .mean()
    .sort_values("chain_len")
)

# -----------------------------
# Plot 1: runtime vs number of atoms
# -----------------------------
plt.figure(figsize=(10, 6))

plt.plot(gpu_mean["chain_len"], gpu_mean["label_time_ms"], marker="o", label="GPU label")
plt.plot(gpu_mean["chain_len"], gpu_mean["build_time_ms"], marker="o", label="GPU sort/build")
plt.plot(gpu_mean["chain_len"], gpu_mean["force_time_ms"], marker="o", label="GPU force")
plt.plot(gpu_mean["chain_len"], gpu_mean["total_time_ms"], marker="o", label="GPU total")
plt.plot(cpu_mean["chain_len"], cpu_mean["total_time_ms"], marker="o", label="CPU total")

plt.xscale("log")
plt.yscale("log")
plt.xlabel("Number of atoms")
plt.ylabel("Runtime (ms)")
plt.title("Runtime vs Number of Atoms")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.show()

# -----------------------------
# Plot 2: speedup vs number of atoms
# speedup = CPU total / GPU total
# -----------------------------
merged = pd.merge(
    gpu_mean[["chain_len", "total_time_ms"]],
    cpu_mean[["chain_len", "total_time_ms"]],
    on="chain_len",
    suffixes=("_gpu", "_cpu")
)

merged["speedup"] = merged["total_time_ms_cpu"] / merged["total_time_ms_gpu"]

plt.figure(figsize=(10, 6))
plt.plot(merged["chain_len"], merged["speedup"], marker="o")

plt.xscale("log")
plt.xlabel("Number of atoms")
plt.ylabel("Speedup (CPU time / GPU time)")
plt.title("Speedup vs Number of Atoms")
plt.grid(True, which="both", alpha=0.3)
plt.show()

## Simulation Run

In [13]:
@cuda.jit
def recover_kernel(recover_array, candidate_array, org_idx, n):
    idx = cuda.grid(1)
    if idx >= n:
        return
    original_position = org_idx[idx]
    recover_array[0, original_position] = candidate_array[0, idx]
    recover_array[1, original_position] = candidate_array[1, idx]
    recover_array[2, original_position] = candidate_array[2, idx]

In [14]:
def rescale_d(velocities: cp.ndarray, mass: float, temperature: float):
  n = velocities.shape[1]
  ke = 1/2*cp.sum(mass * (cp.linalg.norm(velocities, axis=0)**2))
  cur_temp = 2/3 * ke / n
  velocities *= cp.sqrt(temperature / cur_temp)

def rescale_h(velocities: np.ndarray, mass: float, temperature: float):
  n = velocities.shape[1]
  ke = 1/2*np.sum(mass * (np.linalg.norm(velocities, axis=0)**2))
  cur_temp = 2/3 * ke / n
  velocities *= np.sqrt(temperature / cur_temp)

def write_traj(filename, frames):
  n = frames[0].shape[1]
  with open(filename, "w") as f:
    for pos in frames:
      f.write(f"{n}\nframe\n")
      for x, y, z in zip(pos[0], pos[1], pos[2]):
        if x<0 or y<0 or z<0:
          print(f"({x}, {y}, {z})")
        f.write(f"CG {x:.6f} {y:.6f} {z:.6f}\n")
        
def read_traj(traj):
  n_frames = 0
  with open(traj, 'r') as file:
    n_atoms = int(file.readline())
    for line in file:
      if line.strip().lower() == 'frame':
        n_frames += 1
      
  pos_line = re.compile(r'\w+\s+(\d+.\d+)\s+(\d+.\d+)\s+(\d+.\d+)')
  frames = np.zeros((n_frames, 3, n_atoms))
  with open(traj, 'r') as file:
    for frame in range(n_frames):
      file.readline() # reads atom number
      file.readline() # reads "frame"
      for atom_idx in range(n_atoms):
        line = file.readline().strip()
        pos = pos_line.search(line)
        for i in range(SPACE_N_DIM):
          x = float(pos.group(i+1))
          frames[frame, i, atom_idx] = x
  return frames

In [ ]:
# host function
def run_md_cutoff(
  positions: cp.ndarray | np.ndarray, # 3 * n
  velocities: cp.ndarray | np.ndarray, # 3 * n
  dt: float,
  mass: float,
  temperature: float,
  steps: int,
  box_size: float,
  rescale_interval: int,
  save_interval: int,
  sorted = False,
  device = False
):
  frames = np.zeros(shape = ((steps-1)//save_interval+1, 3, positions.shape[1]), dtype = np.float64)

  if sorted and device:
    positions_recovered = cp.zeros(shape = (3, positions.shape[1]), dtype = cp.float64)
    org_idx = cp.arange(positions.shape[1], dtype=cp.int32)

  if device:
    forces = cp.zeros(shape = (3, positions.shape[1]), dtype = cp.float64)
  else:
    forces = np.zeros(shape = (3, positions.shape[1]), dtype = np.float64)

  if sorted and device:
    velocities = velocities[:, org_idx]
  
  velocities += 0.5 * forces / mass * dt

  positions += velocities * dt
  if device:
    positions -= box_size * cp.floor(positions / box_size)
  else:
    positions -= box_size * np.floor(positions / box_size)
  if device:
    frames[0] = positions.get()
  else:
    frames[0] = positions.copy()

  for step in tqdm(range(1, steps)):
    if device:
        if sorted:
          order = calc_force_cutoff_gpu_sorted_wrapper(forces, positions, org_idx)
          velocities = velocities[:, order]
        else:
          calc_force_cutoff_gpu_unsorted_wrapper(forces, positions)
    else:
      calc_force_cutoff_sequential_org_wrapper(forces, positions)
    dv = forces / mass * dt

    if (step-1) % rescale_interval == 0:
      velocities += 0.5 * dv
      if device:
        rescale_d(velocities, mass, temperature)
      else:
        rescale_h(velocities, mass, temperature)
      velocities += 0.5 * dv
    else:
      velocities += dv

    positions += velocities * dt
    if device:
      positions -= box_size * cp.floor(positions / box_size)
    else:
      positions -= box_size * np.floor(positions / box_size)
  
    if step % save_interval == 0:
      if device:
        if sorted:
          n = positions.shape[1]
          grid_size = (n + CUTOFF_BLOCK_SIZE - 1) // CUTOFF_BLOCK_SIZE
          recover_kernel[grid_size, CUTOFF_BLOCK_SIZE](positions_recovered, positions, org_idx, n)
          frames[step // save_interval] = positions_recovered.get()
        else:
          frames[step // save_interval] = positions.get()
      else:
        frames[step // save_interval] = positions.copy()
  
  return frames

In [68]:
n_atoms = 1000
k = 500
r0 = 1
box_size = n_atoms * r0 * 2.5
epsilon_attractive = 0.5
epsilon_repulsive = 1.0
sigma = 1.0
const_params = np.array([box_size, k, r0, epsilon_attractive, epsilon_repulsive, sigma])

seed = 42
rng_np = np.random.default_rng(seed)
rng_cp = cp.random.default_rng(seed)

dt = 0.01
mass = 1
temperature = 0.5
steps = 1001
rescale_interval = 100
save_interval = 10

pos = initialize_chain_numba(n_atoms, box_size, r0, rng_np, dtype = np.float64)
v = initialize_velocities_cupy(n_atoms, target_temperature=temperature, mass=mass, rng = rng_cp)


In [69]:
positions_d = cp.array(pos)
velocities_d = v.copy()
frames_cg = run_md_cutoff(
  positions_d, velocities_d, dt, mass, temperature,
  steps, box_size,
  rescale_interval,
  save_interval,
  sorted = True,
  device = True
)
write_traj("b", frames_cg)

  0%|          | 0/1000 [00:00<?, ?it/s]

In [70]:
velocities_h = v.get()
frames_cc = run_md_cutoff(
  pos, velocities_h, dt, mass, temperature,
  steps, box_size,
  rescale_interval,
  save_interval,
  sorted = False,
  device = False
)
write_traj("a", frames_cc)

  0%|          | 0/1000 [00:00<?, ?it/s]

In [71]:
frames_ccs = read_traj("a")
frames_cgs = read_traj("b")

In [72]:
np.all(frames_cgs == frames_ccs, axis=-1)

array([[ True,  True,  True],
       [ True,  True,  True],
       [ True,  True,  True],
       [ True,  True,  True],
       [ True,  True,  True],
       [ True,  True,  True],
       [ True,  True,  True],
       [ True,  True,  True],
       [ True,  True,  True],
       [ True,  True,  True],
       [ True,  True,  True],
       [ True,  True,  True],
       [ True,  True,  True],
       [ True,  True,  True],
       [ True,  True,  True],
       [ True,  True,  True],
       [ True,  True,  True],
       [ True,  True,  True],
       [ True,  True,  True],
       [ True,  True,  True],
       [ True,  True,  True],
       [ True,  True, False],
       [ True,  True, False],
       [False, False, False],
       [False, False, False],
       [False, False, False],
       [False, False, False],
       [False, False, False],
       [False, False, False],
       [False, False, False],
       [False, False, False],
       [False, False, False],
       [False, False, False],
       [Fa

In [74]:
print(frames_cgs[:-1])
print(frames_ccs[:-1])
print(np.allclose(frames_cgs[:-1], frames_ccs[:-1], atol=1e-5))

[[[1249.993254 1250.229565 1250.5981   ... 1235.766018 1236.321593
   1236.010529]
  [1250.005048 1249.22467  1248.437926 ... 1244.902937 1244.903228
   1244.37434 ]
  [1249.998728 1250.569957 1250.049047 ... 1241.230599 1242.060649
   1241.267259]]

 [[1732.268645 1011.504818 1253.855771 ... 1224.940066 1236.321649
   1247.52948 ]
  [2485.183576 1444.832218 1252.221381 ... 1268.306876 1244.898265
   1223.67156 ]
  [1711.96284   860.451206 1242.69659  ... 1239.606304 1242.060538
   1245.116716]]

 [[2214.544035  772.780072 1257.104109 ... 1214.155941 1236.325176
   1259.048595]
  [1220.362104 1640.439766 1255.860169 ... 1291.667544 1244.884464
   1202.963488]
  [2173.926951  470.332454 1235.405021 ... 1237.992232 1242.06262
   1248.976918]]

 ...

 [[1072.74716  1362.982098 1283.090735 ... 1127.882938 1236.162104
   1351.201514]
  [1101.790327  705.300148 1284.970069 ... 1478.552881 1244.871878
   1037.29891 ]
  [ 869.639841 2349.382442 1177.072568 ... 1225.079661 1241.933942
   1279.8